# prefkit run_inference (Colab or Kaggle T4)

One runtime = one `MODEL_SLOT`. Headline backend **hf**. 4-bit NF4, float16.

Knobs in the config cell: `FRAME` (`default` / `empty` / `persona` / `custom`), optional `SYSTEM` (custom wrap), `METHODS` (`None`/`all` / `M1` / `M1,M3`), `SEED`, `TAG`.

In [ ]:
import os
# Colab/Kaggle: clone then cd. Local: skip if already at repo root.
if not os.path.exists("prefkit"):
    !git clone https://github.com/Kaustubh73/prefkit.git
    %cd prefkit
!nvidia-smi

In [ ]:
!pip install -e ".[hf]"

In [ ]:
import os
os.environ["PREFKIT_BACKEND"] = "hf"   # required for CMS/E3
MODEL_SLOT = "S"          # S | E3_M | M_2507 | L | XL | cross
FRAME = "default"         # default | empty | persona | custom
SYSTEM = None             # None → frozen FRAME string; str → custom (frame recorded as custom)
OUTCOMES = "data/outcomes.json"  # E1 N=24. Debug: data/outcomes.smoke.json
SEED = None               # None → decode.yaml; or int
METHODS = None            # None or "all" → M1–M4; "M1" → one; "M1,M3" → few
TAG = None                # None → no suffix; or "seed1" / "m4" / "auditor"

In [ ]:
from prefkit.cli import cmd_run
import argparse
args = argparse.Namespace(
    slot=MODEL_SLOT,
    frame=FRAME,
    outcomes=OUTCOMES,
    seed=SEED,
    methods=METHODS,
    tag=TAG,
    system=SYSTEM,
)
cmd_run(args)

In [ ]:
from pathlib import Path
import yaml
from prefkit.cli import _resolve_system, _result_path

models = yaml.safe_load(Path("configs/models.yaml").read_text())
hf_id = models[MODEL_SLOT]["hf"]
frame, _ = _resolve_system(FRAME, SYSTEM)
path = str(_result_path(hf_id, frame, OUTCOMES, TAG))
print("wrote", path)
try:
    from google.colab import files
    files.download(path)
except ImportError:
    try:
        from IPython.display import FileLink, display
        display(FileLink(path))
    except Exception:
        print("download the JSON from results/")